In [7]:
seasons = 2020,2021

races_per_season = {2020: 17, 2021: 22}

all_laps = []
all_weather = []
all_trackStatus = []

In [8]:
import logging
import fastf1
from pathlib import Path


logging.getLogger('fastf1').setLevel(logging.CRITICAL)
fastf1.Cache.enable_cache('cache')

In [9]:
import time
for season_year in seasons:

    for round_num in range(1, races_per_season[season_year] + 1):
        try:
            print("Sleeping to avoid rate limit...")
            time.sleep(1.5)
            print(f"Loading {season_year} - Round {round_num}")
            session = fastf1.get_session(season_year, round_num, "R")

            session.load(laps=True, telemetry=True, weather=True, messages=True)

            laps = session.laps.copy()
            laps["Season"] = season_year
            laps["Round"] = round_num
            laps['Date'] = session.date + laps['LapStartTime']
            laps = laps[[
                'Date', 'Time', 'LapStartTime', 'PitOutTime', 'PitInTime', 'LapNumber',
                'TyreLife', 'Stint', 'Compound','LapTime', 'Position', 'TrackStatus',
                'Season', 'Round', 'DriverNumber', 'Driver',
                'Sector1Time', 'Sector2Time', 'Sector3Time',
                'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST',
                'FreshTyre', 'Team',
            ]]

            all_laps.append(laps)

            weather = session.weather_data.copy()
            weather["Season"] = season_year
            weather["Round"] = round_num
            if 'Date' not in weather.columns:
                 weather['Date'] = session.date + weather['Time']
            all_weather.append(weather)

            trackStatus = session.track_status.copy()
            trackStatus["Season"] = season_year
            trackStatus["Round"] = round_num
            trackStatus['Date'] = session.date + trackStatus['Time']
            all_trackStatus.append(trackStatus)

        except Exception as e:
            print(f"Error loading {season_year} Round {round_num}: {e}")



Sleeping to avoid rate limit...
Loading 2020 - Round 1
Sleeping to avoid rate limit...
Loading 2020 - Round 2
Sleeping to avoid rate limit...
Loading 2020 - Round 3
Sleeping to avoid rate limit...
Loading 2020 - Round 4
Sleeping to avoid rate limit...
Loading 2020 - Round 5
Sleeping to avoid rate limit...
Loading 2020 - Round 6
Sleeping to avoid rate limit...
Loading 2020 - Round 7
Sleeping to avoid rate limit...
Loading 2020 - Round 8
Sleeping to avoid rate limit...
Loading 2020 - Round 9
Sleeping to avoid rate limit...
Loading 2020 - Round 10
Sleeping to avoid rate limit...
Loading 2020 - Round 11
Sleeping to avoid rate limit...
Loading 2020 - Round 12
Sleeping to avoid rate limit...
Loading 2020 - Round 13
Sleeping to avoid rate limit...
Loading 2020 - Round 14
Sleeping to avoid rate limit...
Loading 2020 - Round 15
Sleeping to avoid rate limit...
Loading 2020 - Round 16
Sleeping to avoid rate limit...
Loading 2020 - Round 17
Sleeping to avoid rate limit...
Loading 2021 - Round 1
Sl

In [35]:
import pandas as pd
from pathlib import Path

if all_laps and all_weather and all_trackStatus:
    output_dir = Path("csv_data")
    output_dir.mkdir(parents=True, exist_ok=True) # Ensure directory exists

    print("Concatenating data...")
    laps_df = pd.concat(all_laps, ignore_index=True)
    weather_df = pd.concat(all_weather, ignore_index=True)
    track_status_df = pd.concat(all_trackStatus, ignore_index=True)

    for df in [laps_df, weather_df, track_status_df]:
        df['Date'] = pd.to_datetime(df['Date'])

    laps_df = laps_df.sort_values('Date')
    weather_df = weather_df.sort_values('Date')
    track_status_df = track_status_df.sort_values('Date')

    print("Saving separate CSV files...")
    laps_df.to_csv(output_dir / "laps_all.csv", index=False)
    weather_df.to_csv(output_dir / "weather_all.csv", index=False)
    track_status_df.to_csv(output_dir / "track_status_all.csv", index=False)

    print("Merging data for final file...")

    laps_with_weather = pd.merge_asof(
        laps_df,
        weather_df,
        on='Date',
        by=['Season', 'Round'],
        direction='backward',
        suffixes=('', '_weather')
    )

    track_status_df = track_status_df.rename(columns={'Status': 'TrackStatus_Code'})
    ts_cols = ['Date', 'Season', 'Round', 'TrackStatus_Code']

    final_df = pd.merge_asof(
        laps_with_weather,
        track_status_df[ts_cols],
        on='Date',
        by=['Season', 'Round'],
        direction='backward'
    )

    final_df.to_csv(output_dir / "final_merged_data.csv", index=False)
    print(" - final_merged_data.csv saved")
    print("Done!")

else:
    print("Error: One or more data lists are empty. Check your loop.")

Error: One or more data lists are empty. Check your loop.
